# Pairs Trading

For this model, I want the model to take into account N crypto currencies and re-balance a portfolio for some frequency based on a pairs trading strategy. I want the strategy to dynamically search over all the currencies and group them into pairs if they are correlated by a certain amount, disregarding pairs that have the lowest correlation.

For this model I am going to modify an existing model defined in the paper [MISG 2007](https://miis.maths.ox.ac.uk/129/1/misg2007paper3.pdf) that describes a jump-diffusion mean-reverting model. This model follows the equation:

$$
\partial S_t = \alpha (S^* - \ln{S_t})S_t\partial t + S_t \sigma \partial Z_t + S_t K \partial q_t
$$

where $\alpha$ is the mean-reversion rate, $S^*$ is the mean-reversion level, $\sigma$ the volatility of the spot price, $K$ is the jump size which may, for example be taken to follow a normal distribution $N(\mu,v^2)$ and $\partial q$ is the Poisson process so that

$$
\partial q_t = 1 \text{ with probability } \lambda \partial t, \partial q_t = 0 \text{ with probability } 1 - \lambda \partial t
$$

(where $\lambda$ is the average number of jumps per year)

Instead of reverting to $S^*$, we can just use the other random price at time $t$ so that the price will tend to revert towards the price of the other currency. We can then re-write the above as:


$$
\partial S_t^2 = \alpha (S_t^1 - \ln{S_t^2})S_t^2\partial t + S_t^2 \sigma \partial Z_t + S_t^2 K \partial q_t
$$

And the reverse:

$$
\partial S_t^1 = \alpha (S_t^2 - \ln{S_t^1})S_t^1\partial t + S_t^1 \sigma \partial Z_t + S_t^1 K \partial q_t
$$

For a pair of two currencies $S^1$ and $S^2$.



In [ ]:
from tqdm.notebook import tqdm
import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

N_sims = 50
N_days = 365

start = dt.datetime.now()
end = start + dt.timedelta(days=N_days)
T = np.arange(start, end, np.timedelta64(1, '1m'))

# Brownian motion model
S_0 = 100
epsilon = np.random.normal(0, 1, (N_sims, len(T)))
sigma = np.ones((N_sims, len(T))) * 0.075
delta_t = np.ones((N_sims, len(T))) * (1 / len(T))
mu = np.ones((N_sims, len(T))) * 0.025
S_brownian = mu * delta_t + sigma * epsilon * np.sqrt(delta_t)

# Jump-diffusion model
theta = 1
lam = 50 / len(T)
jump = np.random.poisson(lam=lam, size=(N_sims, len(T)))
S_jump = jump * theta * delta_t

# Mean-reverting model
alpha = 0.05
S_m = 100
eta = 0.025
S_mean_reverting = eta * (S_m * np.exp(-alpha * delta_t) - S_0) * delta_t

# Jump-diffusion model
S = S_0 * np.cumprod(1 + S_brownian + S_jump + S_mean_reverting, axis=1)

# Plot the results
N_plots = 50
for i in tqdm(np.linspace(0, N_sims - 1, N_plots).astype(int)):
    plt.gca().xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.gca().xaxis.set_major_locator(mdates.DayLocator(interval=10))  # Show every 10th day
    plt.xticks(rotation=45)
    plt.plot(T, S[i, :])
plt.show()